# 3. Extract phase features — corrected

Mean skin temperature is calculated using the four-site ISO weighting stated in the manuscript. Test-relative descriptive changes use the mean of the first 60 s of the initial standing phase. The phase-specific baseline used for Table 3 remains a separate analysis in notebook 6.


In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Project root not found. Run the notebook from inside the project repository.")

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def report_dir(root: Path, analysis_name: str, subfolder: str | None = None) -> Path:
    base = root / "reports" / analysis_name
    return ensure_dir(base if subfolder is None else base / subfolder)

ROOT = find_project_root()
print("ROOT:", ROOT)


ROOT: /home/eleonorab/repository/convective_cooling


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd


ROOT = find_project_root()
INPUT_FILE = ROOT / "data" / "processed" / "analysis" / "analysis_dataset_wide.csv"
OUTPUT_DIR = ROOT / "data" / "processed" / "features"
ensure_dir(OUTPUT_DIR)

PHASE_ORDER = [
    "standing_1", "walk_low_1", "walk_mod_1",
    "walk_low_2", "walk_mod_2", "standing_2",
]

SKIN_TEMPERATURE_WEIGHTS = {
    "t_neck": 0.28,
    "t_scapula": 0.28,
    "t_hand": 0.16,
    "t_shin": 0.28,
}

TEST_BASELINE_SECONDS = 60.0


def load_data() -> pd.DataFrame:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Missing input file: {INPUT_FILE}")
    data = pd.read_csv(INPUT_FILE)
    for column in ["elapsed_minutes", "elapsed_seconds", *SKIN_TEMPERATURE_WEIGHTS]:
        if column in data:
            data[column] = pd.to_numeric(data[column], errors="coerce")
    data["condition"] = data["condition"].astype(str).str.strip().str.lower()
    data["subject_id"] = data["subject_id"].astype(str).str.strip()
    data["phase"] = data["phase"].astype(str).str.strip()
    return data


def add_mean_skin_temperature(data: pd.DataFrame) -> pd.DataFrame:
    missing = [column for column in SKIN_TEMPERATURE_WEIGHTS if column not in data]
    if missing:
        raise KeyError(
            "The ISO-weighted mean skin temperature requires all four sites. "
            f"Missing columns: {missing}. Check notebook 1 and the raw signal files."
        )

    data = data.copy()
    weighted_terms = [
        weight * data[column]
        for column, weight in SKIN_TEMPERATURE_WEIGHTS.items()
    ]
    # Ordinary addition intentionally returns NaN if any required site is missing.
    data["mean_skin_temperature"] = sum(weighted_terms)
    return data


def get_signal_columns(data: pd.DataFrame) -> list[str]:
    excluded = {
        "subject_id", "condition", "timestamp", "elapsed_seconds",
        "elapsed_seconds_bin", "elapsed_minutes", "phase", "time_bin_5min",
    }
    return [column for column in data.columns if column not in excluded]


def compute_test_baselines(data: pd.DataFrame, signals: list[str]) -> pd.DataFrame:
    baseline_window = data.loc[
        (data["phase"] == "standing_1")
        & (data["elapsed_seconds"] <= TEST_BASELINE_SECONDS)
    ].copy()

    rows = []
    for (subject, condition), group in baseline_window.groupby(
        ["subject_id", "condition"], observed=True
    ):
        for signal in signals:
            values = pd.to_numeric(group[signal], errors="coerce")
            rows.append({
                "subject_id": subject,
                "condition": condition,
                "signal": signal,
                "baseline_value": values.mean(),
                "baseline_n": int(values.notna().sum()),
            })
    return pd.DataFrame(rows)


def compute_slope(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 2:
        return np.nan
    return float(np.polyfit(x[valid], y[valid], 1)[0])


def extract_phase_features(
    data: pd.DataFrame,
    signals: list[str],
    baselines: pd.DataFrame,
) -> pd.DataFrame:
    lookup = baselines.set_index(
        ["subject_id", "condition", "signal"]
    )["baseline_value"].to_dict()

    rows = []
    for (subject, condition, phase), group in data.groupby(
        ["subject_id", "condition", "phase"], observed=True
    ):
        group = group.sort_values("elapsed_minutes")
        for signal in signals:
            values = pd.DataFrame({
                "time": pd.to_numeric(group["elapsed_minutes"], errors="coerce"),
                "value": pd.to_numeric(group[signal], errors="coerce"),
            }).dropna()
            if values.empty:
                continue

            baseline = lookup.get((subject, condition, signal), np.nan)
            mean_value = values["value"].mean()
            rows.append({
                "subject_id": subject,
                "condition": condition,
                "phase": phase,
                "signal": signal,
                "n_samples": len(values),
                "mean": mean_value,
                "std": values["value"].std(),
                "median": values["value"].median(),
                "min": values["value"].min(),
                "max": values["value"].max(),
                "delta_in_out": values["value"].iloc[-1] - values["value"].iloc[0],
                "slope": compute_slope(
                    values["time"].to_numpy(float), values["value"].to_numpy(float)
                ),
                "baseline_value": baseline,
                "delta_mean_from_baseline": (
                    mean_value - baseline if pd.notna(baseline) else np.nan
                ),
            })

    output = pd.DataFrame(rows)
    if not output.empty:
        output["phase"] = pd.Categorical(
            output["phase"], categories=PHASE_ORDER, ordered=True
        )
        output = output.sort_values(
            ["signal", "condition", "subject_id", "phase"]
        ).reset_index(drop=True)
    return output


def main() -> None:
    data = add_mean_skin_temperature(load_data())
    signals = get_signal_columns(data)
    baselines = compute_test_baselines(data, signals)
    phase_features = extract_phase_features(data, signals, baselines)

    data.to_csv(OUTPUT_DIR / "analysis_dataset_with_tsk.csv", index=False)
    baselines.to_csv(OUTPUT_DIR / "baseline_values.csv", index=False)
    phase_features.to_csv(OUTPUT_DIR / "phase_features_long.csv", index=False)

    print("Saved:")
    print(OUTPUT_DIR / "analysis_dataset_with_tsk.csv")
    print(OUTPUT_DIR / "baseline_values.csv")
    print(OUTPUT_DIR / "phase_features_long.csv")
    print("Mean-skin-temperature valid rows:", data["mean_skin_temperature"].notna().sum())


if __name__ == "__main__":
    main()


Saved:
/home/eleonorab/repository/convective_cooling/data/processed/features/analysis_dataset_with_tsk.csv
/home/eleonorab/repository/convective_cooling/data/processed/features/baseline_values.csv
/home/eleonorab/repository/convective_cooling/data/processed/features/phase_features_long.csv
Mean-skin-temperature valid rows: 8795
